In [1]:
%pip -q install rdkit pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 53.9 MB/s eta 0:00:00


In [2]:
import io
import json
import math
import hashlib
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from rdkit import Chem, rdBase
from rdkit.Contrib.SA_Score import sascorer
from google.colab import files
from IPython.display import display

# Check that the installed scorer and fragment weights work.
test_score = sascorer.calculateScore(Chem.MolFromSmiles("CCO"))
assert test_score is not None and 1 <= test_score <= 10
print("SA scorer dependency check: PASS")

print("Upload the broad v3 library ZIP.")
uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith(".zip")]
assert len(zip_names) == 1, "Upload one library ZIP."

input_bytes = uploaded[zip_names[0]]

with zipfile.ZipFile(io.BytesIO(input_bytes)) as archive:
    table = pd.read_csv(
        io.BytesIO(archive.read("tryptanthrin_analogs_clean.csv")),
        keep_default_na=False
    )

required = {"Analog_ID", "SMILES", "Category"}
assert required.issubset(table.columns)
assert len(table) == 11393, "Unexpected input size; check the library version."
assert table["Analog_ID"].is_unique
assert table["SMILES"].is_unique
assert table["Analog_ID"].ne("").all()
assert table["SMILES"].ne("").all()

# Preserve input values and order.
scored = table.copy()
scores, statuses, errors = [], [], []

for number, row in enumerate(table.itertuples(), start=1):
    try:
        mol = Chem.MolFromSmiles(row.SMILES)
        if mol is None:
            raise ValueError("SMILES could not be parsed.")

        score = sascorer.calculateScore(mol)
        if score is None or not math.isfinite(score):
            raise ValueError("No finite SA score returned.")
        if not 1 <= score <= 10:
            raise ValueError("SA score outside expected range.")

        scores.append(float(score))
        statuses.append("PASS")
        errors.append("")

    except Exception as error:
        scores.append(float("nan"))
        statuses.append("FAILED")
        errors.append(str(error))

    if number % 2000 == 0:
        print(f"Scored {number:,}/{len(table):,}")

scored["SA_score"] = scores
scored["SA_status"] = statuses
scored["SA_error"] = errors

assert len(scored) == len(table)
assert scored["Analog_ID"].equals(table["Analog_ID"])
assert scored["SMILES"].equals(table["SMILES"])

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
out = Path("SA_scoring_runs") / run_id
out.mkdir(parents=True, exist_ok=False)

scored.to_csv(out / "all_analogs_with_SA.csv", index=False)

scored[[
    "Analog_ID", "SMILES", "Category",
    "SA_score", "SA_status", "SA_error"
]].to_csv(out / "SA_scores.csv", index=False)

failed = scored.loc[scored["SA_status"] == "FAILED"]
failed.to_csv(out / "SA_failures.csv", index=False)

successful = scored.loc[scored["SA_status"] == "PASS"]
summary = pd.DataFrame([{
    "Input_structures": len(table),
    "Successfully_scored": len(successful),
    "Scoring_failures": len(failed),
    "SA_min": successful["SA_score"].min(),
    "SA_median": successful["SA_score"].median(),
    "SA_max": successful["SA_score"].max(),
    "Compounds_excluded": 0
}])
summary.to_csv(out / "SA_summary.csv", index=False)

# Preserve scorer implementation and weights for reproducibility.
scorer_path = Path(sascorer.__file__)
weights_path = scorer_path.with_name("fpscores.pkl.gz")
assert weights_path.exists()

scorer_dir = out / "scorer"
scorer_dir.mkdir()
shutil.copy2(scorer_path, scorer_dir / scorer_path.name)
shutil.copy2(weights_path, scorer_dir / weights_path.name)

metadata = {
    "Run_UTC": run_id,
    "Input_filename": zip_names[0],
    "Input_ZIP_SHA256": hashlib.sha256(input_bytes).hexdigest(),
    "RDKit_version": rdBase.rdkitVersion,
    "pandas_version": pd.__version__,
    "Scorer": "rdkit.Contrib.SA_Score.sascorer",
    "Scorer_SHA256": hashlib.sha256(
        scorer_path.read_bytes()
    ).hexdigest(),
    "Fragment_weights_SHA256": hashlib.sha256(
        weights_path.read_bytes()
    ).hexdigest(),
    "Method_reference": "Ertl and Schuffenhauer (2009)",
    "Interpretation": "Lower scores indicate lower estimated difficulty.",
    "Filtering_applied": False,
    "Limitation": "No synthesis route or experimental feasibility established."
}
(out / "SA_run_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)

display(summary)
display(successful.groupby("Category")["SA_score"].agg(
    ["count", "min", "median", "max"]
))
display(scored.loc[
    scored["Category"] == "Parent",
    ["Analog_ID", "SA_score", "SA_status"]
])

archive_path = shutil.make_archive(
    f"/content/tryptanthrin_SA_scores_{run_id}",
    "zip",
    root_dir=str(out)
)
files.download(archive_path)

SA scorer dependency check: PASS
Upload the broad v3 library ZIP.


Saving broad_tryptanthrin_library_20261020T061234342390Z.zip to broad_tryptanthrin_library_20261020T061234342390Z.zip
Scored 2,000/11,393
Scored 4,000/11,393
Scored 6,000/11,393
Scored 8,000/11,393
Scored 10,000/11,393


,Input_structures,Successfully_scored,Scoring_failures,SA_min,SA_median,SA_max,Compounds_excluded
0,11393,11393,0,1.938639,2.483888,2.844676,0


,count,min,median,max
Category,,,,
Exploratory_combination,58,2.145926,2.301428,2.486233
Exploratory_design,11297,2.064828,2.485476,2.844676
Parent,1,1.938639,1.938639,1.938639
Published_reference,37,2.038331,2.212478,2.506207


,Analog_ID,SA_score,SA_status
0,TRY_14d107e8e7b92eda,1.938639,PASS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>